# facproject — Colab Bootstrap

Этот ноутбук клонирует репозиторий, устанавливает зависимости и запускает нужную стадию пайплайна.

**Как использовать:**
1. Открой в VS Code, нажми `Select Kernel` → `Colab` → выбери рантайм (T4 бесплатно)
2. Запусти ячейки по порядку
3. Для реальных данных — смонтируй Google Drive (секция ниже)

In [ ]:
# 1. Проверка окружения
import sys, os
print('Python:', sys.version)
print('GPU:', end=' ')
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || print('нет GPU')
print('Workdir:', os.getcwd())

In [ ]:
# 2. Клонирование репозитория
# Для приватного репо: замени TOKEN на GitHub Personal Access Token
# (Settings → Developer Settings → Fine-grained tokens, scope: Contents read)
import os

TOKEN = os.environ.get('GITHUB_TOKEN', '')  # задай через Colab Secrets или вставь напрямую
REPO = 'hudyakovictor/facproject'

if TOKEN:
    url = f'https://{TOKEN}@github.com/{REPO}.git'
else:
    url = f'https://github.com/{REPO}.git'

if not os.path.exists('/content/facproject'):
    !git clone {url} /content/facproject
else:
    print('Репо уже клонировано, pulling...')
    !cd /content/facproject && git pull

os.chdir('/content/facproject')
print('CWD:', os.getcwd())

In [ ]:
# 3. Установка зависимостей
!pip install -q -r app6/api/requirements.txt
# dev-инструменты (pytest, ruff, httpx) — опционально:
# !pip install -q -r requirements-dev.txt
print('✅ Зависимости установлены')

In [ ]:
# 4. [ОПЦИОНАЛЬНО] Монтирование Google Drive для реальных данных
# Раскомментируй если у тебя данные загружены на Google Drive

# from google.colab import drive
# drive.mount('/content/drive')

# После монтирования укажи переменные:
# import os
# os.environ['DEEPUTIN_STORAGE_ROOT'] = '/content/drive/MyDrive/facproject_data/storage'
# os.environ['DEEPUTIN_CALIBRATION_ROOT'] = '/content/drive/MyDrive/facproject_data/calibration'

print('⚠️  Google Drive не смонтирован — используем fixture-режим')

In [ ]:
# 5. Генерация синтетических данных Stage 1 (fixture-режим, без весов и SD-CARD)
!python -m app6.stage2_v2.fixtures /tmp/test_s1 --per-day 2 --change 0.01
print('✅ Фикстуры сгенерированы в /tmp/test_s1')

In [ ]:
# 6. Запуск stage2_v2 (новая версия, рекомендуется)
# Профили: default | strict_evidence | exploratory | fast_smoke
PROFILE = 'fast_smoke'  # самый быстрый для проверки

!python -m stage2_v2.cli run \
    --stage1 /tmp/test_s1 \
    --output /tmp/stage2_out \
    --profile {PROFILE}

print('✅ stage2_v2 завершён, результаты в /tmp/stage2_out')

In [ ]:
# 7. Просмотр результатов
import os, json
output_dir = '/tmp/stage2_out'
files = os.listdir(output_dir) if os.path.exists(output_dir) else []
print(f'Файлов в output: {len(files)}')
for f in sorted(files)[:20]:
    print(' ', f)

In [ ]:
# 8. [С РЕАЛЬНЫМИ ДАННЫМИ] — раскомментируй после монтирования Drive
# Запуск полного пайплайна через RUN_PROJECT.sh

# import subprocess
# result = subprocess.run(
#     ['bash', 'RUN_PROJECT.sh', 'stage2'],
#     capture_output=True, text=True, cwd='/content/facproject'
# )
# print(result.stdout)
# if result.returncode != 0:
#     print('STDERR:', result.stderr)

In [ ]:
# 9. [ОПЦИОНАЛЬНО] Запуск тестов
# !pip install -q pytest httpx
# !python -m pytest -q app6/test_module